In [1]:
pip install -q faiss-cpu sentence-transformers rank-bm25 bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 90.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 86.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face login successful")

Hugging Face login successful


In [3]:
# ==========================================================
# INSTALL GPU LLAMA-CPP (KAGGLE)
# ==========================================================

# !pip uninstall -y llama-cpp-python -q use if cpp is installed


# Enable CUDA build
%env CMAKE_ARGS=-DGGML_CUDA=on
%env FORCE_CMAKE=1

!pip install llama-cpp-python -U \
    --force-reinstall \
    --no-cache-dir \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

env: CMAKE_ARGS=-DGGML_CUDA=on
env: FORCE_CMAKE=1
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 276.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 287.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 276.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: MarkupSafe
    Found existing installation: MarkupSafe 3.0.3
  

In [57]:
import llama_cpp

print(llama_cpp.llama_supports_gpu_offload())

True


In [28]:
!nvidia-smi

Fri May 22 13:44:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             29W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [58]:
#cleaning llm from gpu

import gc
import torch

try:
    del llm
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated() / 1e9)
print(torch.cuda.memory_reserved() / 1e9)

3.628162048
3.644850176


In [2]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="unsloth/gemma-4-31B-it-GGUF",
    filename="gemma-4-31B-it-Q3_K_S.gguf",

    n_gpu_layers=-1,
    split_mode=1,
    main_gpu=0,

    n_ctx=10000,
    n_batch=512,
    flash_attn=True,
    verbose=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./gemma-4-31B-it-Q3_K_S.gguf:   0%|          | 0.00/13.2G [00:00<?, ?B/s]

llama_context: n_ctx_seq (10240) < n_ctx_train (262144) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


# Submission 11. Using approach 2 preprocessing and 31b model


In [ ]:
## fusion dia merge (submssion 4, 6, 7 e eta use kora hoise)

import faiss
import pickle
import torch
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

# ----------------------------------------------------------
# PATHS
# ----------------------------------------------------------
TEST_PATH = "/kaggle/input/competitions/are-you-sure-llm-is-enough-intra-cuet-ml-contest-2-0/Dataset/test.csv"



CHUNKS_PATH = "/kaggle/input/models/nirjharami/sub8-chunk/other/default/1/chunks_recommended2.pkl" ### sub 8 er ta dia try kora jai
EMBED_PATH = "/kaggle/input/models/nirjharami/sub8-embed/other/default/1/bge_m3_embeddings_recommended2.npy"

# ----------------------------------------------------------
# LOAD CHUNKS
# ----------------------------------------------------------
print("Loading chunks...")
with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)
print("Chunks loaded:", len(chunks))

# ----------------------------------------------------------
# LOAD EMBEDDINGS
# ----------------------------------------------------------
print("Loading BGE-M3 embeddings...")
chunk_embeddings = np.load(EMBED_PATH)
print("Embeddings shape:", chunk_embeddings.shape)

# ----------------------------------------------------------
# BM25
# ----------------------------------------------------------
print("Building BM25...")
tokenized_corpus = [c.split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

# ----------------------------------------------------------
# LOAD BGE-M3
# ----------------------------------------------------------
print("Loading BGE-M3 model...")
embed_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cpu"
)
test_embed = embed_model.encode(
    ["test"],
    normalize_embeddings=True
)
print("Model dimension:", test_embed.shape[1])

# ----------------------------------------------------------
# FAISS
# ----------------------------------------------------------
print("Building FAISS index...")
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)
print("FAISS size:", index.ntotal)

# ----------------------------------------------------------
# LOAD RERANKER
# ----------------------------------------------------------
print("Loading reranker...")
reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    device="cuda"
)

# =========================================================
# IMPROVED RETRIEVAL FUNCTION WITH WEIGHTED MERGING
# =========================================================

def retrieve(question,
             bm25_k=5,
             dense_k=15,
             final_k=2,
             bm25_weight=0.7,
             dense_weight=0.3,
             verbose=True):
    """
    Improved retrieval with weighted merging
    + previous chunk augmentation.
    """

    # =====================================================
    # STEP 1: BM25 RETRIEVAL
    # =====================================================
    if verbose:
        print(f"[BM25] Retrieving top {bm25_k}...")

    bm25_scores = bm25.get_scores(question.split())
    bm25_idx = np.argsort(bm25_scores)[::-1][:bm25_k]
    bm25_scores_top = bm25_scores[bm25_idx]

    if verbose:
        print(f"  BM25 scores: {bm25_scores_top}")

    # =====================================================
    # STEP 2: DENSE RETRIEVAL
    # =====================================================
    if verbose:
        print(f"[Dense] Retrieving top {dense_k}...")

    q_embedding = embed_model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    dense_scores, dense_idx = index.search(
        q_embedding,
        dense_k
    )

    dense_idx = dense_idx[0]
    dense_scores = dense_scores[0]

    if verbose:
        print(f"  Dense scores: {dense_scores}")

    # =====================================================
    # STEP 3: WEIGHTED MERGE
    # =====================================================
    if verbose:
        print(
            f"[Merge] Merging with weights "
            f"(BM25={bm25_weight}, Dense={dense_weight})..."
        )

    # Normalize BM25 scores
    if len(bm25_scores_top) > 1:
        bm25_min = bm25_scores_top.min()
        bm25_max = bm25_scores_top.max()

        bm25_norm = (
            (bm25_scores_top - bm25_min)
            / (bm25_max - bm25_min + 1e-8)
        )
    else:
        bm25_norm = np.array([1.0])

    # Normalize Dense scores
    if len(dense_scores) > 1:
        dense_min = dense_scores.min()
        dense_max = dense_scores.max()

        dense_norm = (
            (dense_scores - dense_min)
            / (dense_max - dense_min + 1e-8)
        )
    else:
        dense_norm = np.array([1.0])

    # =====================================================
    # STEP 4: SCORE MERGING
    # =====================================================
    scores_dict = {}

    # BM25
    for idx, score in zip(bm25_idx, bm25_norm):
        scores_dict[int(idx)] = (
            bm25_weight * float(score)
        )

    # Dense
    for idx, score in zip(dense_idx, dense_norm):

        idx_int = int(idx)

        if idx_int in scores_dict:
            scores_dict[idx_int] += (
                dense_weight * float(score)
            )
        else:
            scores_dict[idx_int] = (
                dense_weight * float(score)
            )

    # Sort candidates
    sorted_candidates = sorted(
        scores_dict.items(),
        key=lambda x: x[1],
        reverse=True
    )

    merge_k = max(
        final_k * 2,
        len(sorted_candidates)
    )

    candidate_ids = [
        idx
        for idx, _
        in sorted_candidates[:merge_k]
    ]

    if verbose:
        print(
            f"  Merged {len(candidate_ids)} "
            f"candidates (before reranking)"
        )

        print(
            f"  Merged scores: "
            f"{sorted_candidates[:3]}"
        )

    # =====================================================
    # STEP 5: PREPARE FOR RERANKING
    # =====================================================
    candidate_chunks = [
        (idx, chunks[idx])
        for idx in candidate_ids
    ]

    # =====================================================
    # STEP 6: RERANK
    # =====================================================
    if verbose:
        print(
            f"[Rerank] Reranking "
            f"{len(candidate_chunks)} candidates..."
        )

    pairs = [
        (question, chunk_text)
        for _, chunk_text
        in candidate_chunks
    ]

    rerank_scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidate_chunks, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )

    # =====================================================
    # STEP 7: ADD PREVIOUS CHUNK
    # =====================================================
    best_chunks = []
    best_scores = []

    seen = set()

    for ((idx, chunk_text), score) in ranked[:final_k]:

        merged_parts = []

        # Add previous chunk only
        if idx > 0 and (idx - 1) not in seen:
            merged_parts.append(
                chunks[idx - 1]
            )
            seen.add(idx - 1)

        # Add current chunk
        if idx not in seen:
            merged_parts.append(
                chunk_text
            )
            seen.add(idx)

        merged_chunk = "\n".join(
            merged_parts
        )

        best_chunks.append(
            merged_chunk
        )

        best_scores.append(score)

    # =====================================================
    # STEP 8: FINAL OUTPUT
    # =====================================================
    confidence = np.mean(best_scores)
    confidence = float(
        np.clip(confidence, 0.0, 1.0)
    )

    context = "\n\n".join(
        best_chunks
    )

    if verbose:
        print(
            f"[Result] Selected top "
            f"{final_k} chunks"
        )

        print(
            f"  Rerank scores: "
            f"{best_scores}"
        )

        print(
            f"  Confidence: "
            f"{confidence:.2%}\n"
        )

    return context, confidence


# =========================================================
# ALTERNATIVE: WEIGHTED MERGE WITH RATIO (Simpler)
# =========================================================

def retrieve_v2(question,
                bm25_k=5,
                dense_k=5,
                final_k=3):
    """
    Simpler version using rank-based weighting.
    """
    
    # BM25 retrieval
    bm25_scores = bm25.get_scores(question.split())
    bm25_idx = np.argsort(bm25_scores)[::-1][:bm25_k]
    
    # Dense retrieval
    q_embedding = embed_model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    scores, indices = index.search(q_embedding, dense_k)
    dense_idx = indices[0]
    
    # WEIGHTED MERGE using rank position
    scores_dict = {}
    
    # BM25: earlier rank = higher weight
    for rank, idx in enumerate(bm25_idx):
        scores_dict[int(idx)] = 0.4 * (1.0 - rank / bm25_k)
    
    # Dense: earlier rank = higher weight
    for rank, idx in enumerate(dense_idx):
        idx_int = int(idx)
        dense_score = 0.6 * (1.0 - rank / dense_k)
        if idx_int in scores_dict:
            scores_dict[idx_int] += dense_score
        else:
            scores_dict[idx_int] = dense_score
    
    # Sort and get top candidates
    sorted_candidates = sorted(
        scores_dict.items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    candidate_ids = [idx for idx, _ in sorted_candidates[:final_k*2]]
    candidate_chunks = [chunks[i] for i in candidate_ids]
    
    # RERANK
    pairs = [(question, chunk) for chunk in candidate_chunks]
    rerank_scores = reranker.predict(pairs)
    
    ranked = sorted(
        zip(candidate_chunks, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )
    
    best_chunks = [x[0] for x in ranked[:final_k]]
    context = "\n\n".join(best_chunks)
    confidence = np.mean([x[1] for x in ranked[:final_k]])
    
    return context, confidence


# =========================================================
# USAGE EXAMPLES
# =========================================================

if __name__ == "__main__":
    
    # Example question
    question = "ম্যাক্স ব্যারনের বন্ধুরা তাকে কেন প্রায়ই ঠাট্টা-বিদ্রুপ করে?"
    
    print("=" * 70)
    print("RETRIEVAL WITH WEIGHTED MERGING")
    print("=" * 70)
    
    # Method 1: Score-based weighted merging
    print("\n📌 METHOD 1: Score-based Weighted Merging")
    print("-" * 70)
    context, confidence = retrieve(
        question,
        bm25_k=5,
        dense_k=5,
        final_k=3,
        bm25_weight=0.4,
        dense_weight=0.6
    )
    
    print(f"Context:\n{context}\n")
    print(f"Confidence: {confidence:.2%}\n")
    
    # Method 2: Rank-based weighted merging (simpler)
    print("\n📌 METHOD 2: Rank-based Weighted Merging")
    print("-" * 70)
    context2, confidence2 = retrieve_v2(
        question,
        bm25_k=5,
        dense_k=5,
        final_k=3
    )
    
    print(f"Context:\n{context2}\n")
    print(f"Confidence: {confidence2:.2%}\n")
    
    # Compare which is better
    print("\n" + "=" * 70)
    print("COMPARISON")
    print("=" * 70)
    print(f"Method 1 confidence: {confidence:.2%}")
    print(f"Method 2 confidence: {confidence2:.2%}")
    print(f"Difference: {abs(confidence - confidence2):.2%}")

Loading chunks...
Chunks loaded: 75819
Loading BGE-M3 embeddings...
Embeddings shape: (75819, 1024)
Building BM25...
Loading BGE-M3 model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

RETRIEVAL WITH WEIGHTED MERGING

📌 METHOD 1: Score-based Weighted Merging
----------------------------------------------------------------------
[BM25] Retrieving top 5...
  BM25 scores: [42.86846087 23.64831736 19.11533572 15.50432351 14.50538848]
[Dense] Retrieving top 5...
  Dense scores: [0.57791597 0.5670579  0.5227811  0.5053871  0.5023142 ]
[Merge] Merging with weights (BM25=0.4, Dense=0.6)...
  Merged 8 candidates (before reranking)
  Merged scores: [(25778, 0.9999999283333978), (25780, 0.527914708298363), (29256, 0.16243164539337157)]
[Rerank] Reranking 8 candidates...
[Result] Selected top 3 chunks
  Rerank scores: [np.float32(0.9944946), np.float32(0.14005448), np.float32(0.054799486)]
  Confidence: 39.64%

Context:
সেন্ট লুইস, মিসৌরিতে
একজন মধ্যবয়সী শ্রমিক শ্রেণীর
খাদ্য পরিবেশিকা
(সার‍্যান্ডন)-এর প্রেমে পড়েন। ছবিটির মূল সঙ্গীত স্কোর জর্জ ফেন্টন দ্বারা রচিত হয়েছিল। "একজন তরুণ পুরুষ এবং একজন সাহসী মহিলার গল্প" ("The story of a younger man and a bolder woman") ট্যাগলাইন দিয

In [ ]:
## Approach 2 settings দিয়ে test retrieval save

import pandas as pd
from tqdm.auto import tqdm
import gc
import torch

# =====================================================
# LOAD TEST CSV
# =====================================================

TEST_PATH = "/kaggle/input/competitions/are-you-sure-llm-is-enough-intra-cuet-ml-contest-2-0/Dataset/test.csv"

df = pd.read_csv(TEST_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =====================================================
# STORAGE
# =====================================================

context1_list = []
context2_list = []

confidence1_list = []
confidence2_list = []

# =====================================================
# GENERATE RETRIEVALS
# =====================================================

tqdm_bar = tqdm(
    range(len(df)),
    desc="Generating test contexts"
)

for idx in tqdm_bar:

    try:

        question = str(
            df.iloc[idx]["question"]
        ).strip()

        # =================================================
        # CONTEXT 1
        # submission8 first retrieval
        # =================================================

        context1, confidence1 = retrieve(
            question,
            bm25_k=10,
            dense_k=15,
            final_k=6,
            bm25_weight=0.65,
            dense_weight=0.35,
            verbose=False
        )

        # =================================================
        # CONTEXT 2
        # submission8 retry retrieval
        # =================================================

        context2, confidence2 = retrieve(
            question,
            bm25_k=10,
            dense_k=15,
            final_k=10,
            bm25_weight=0.65,
            dense_weight=0.35,
            verbose=False
        )

        # =================================================
        # STORE
        # =================================================

        context1_list.append(context1)
        context2_list.append(context2)

        confidence1_list.append(
            confidence1
        )

        confidence2_list.append(
            confidence2
        )

        # =================================================
        # SHOW FIRST 3
        # =================================================

        if idx < 3:

            print("\n" + "=" * 100)
            print(f"QUESTION {idx+1}")
            print("=" * 100)

            print("\nQUESTION:")
            print(question)

            print("\nCONTEXT 1:")
            print(context1[:500])

            print("\nCONFIDENCE 1:")
            print(confidence1)

            print("\nCONTEXT 2:")
            print(context2[:500])

            print("\nCONFIDENCE 2:")
            print(confidence2)

        # =================================================
        # TEMP SAVE EVERY 100
        # =================================================

        if (idx + 1) % 100 == 0:

            temp_df = df.iloc[
                :len(context1_list)
            ].copy()

            temp_df["context1"] = (
                context1_list
            )

            temp_df["context2"] = (
                context2_list
            )

            temp_df["confidence1"] = (
                confidence1_list
            )

            temp_df["confidence2"] = (
                confidence2_list
            )

            temp_df.to_csv(
                "test_with_rag_temp.csv",
                index=False
            )

            print(
                f"\n✅ Saved progress:"
                f" {idx+1}/{len(df)}"
            )

        # =================================================
        # MEMORY CLEANUP
        # =================================================

        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:

        print(
            f"\n❌ Error at row {idx}"
        )
        print(e)

        context1_list.append("")
        context2_list.append("")

        confidence1_list.append(0.0)
        confidence2_list.append(0.0)

# =====================================================
# FINAL SAVE
# =====================================================

df["context1"] = context1_list
df["context2"] = context2_list

df["confidence1"] = confidence1_list
df["confidence2"] = confidence2_list

SAVE_NAME = "test_with_rag.csv"

df.to_csv(
    SAVE_NAME,
    index=False
)

print("\n✅ DONE")
print(f"Saved as: {SAVE_NAME}")

print("\nFinal columns:")
print(df.columns.tolist())

print("\nSample:")
print(
    df[
        [
            "index",
            "question",
            "context1",
            "context2"
        ]
    ].head(3)
)

Shape: (1500, 2)
Columns: ['index', 'question']


Generating test contexts:   0%|          | 0/1500 [00:00<?, ?it/s]


QUESTION 1

QUESTION:
অমলক রতন কোহলি কোন ক্ষেত্রে বিশিষ্ট হিসেবে সম্মানিত?

CONTEXT 1:
১৯১৬-এ জন্ম
১৯৯৪-এ মৃত্যু
পশ্চিমবঙ্গের প্রকৌশলী
২০শ শতাব্দীর ভারতীয় উদ্ভাবক
২০শ শতাব্দীর ভারতীয় প্রকৌশলী
ভারতীয় পেটেন্ট ধারক
কলকাতা বিশ্ববিদ্যালয়ের প্রাক্তন শিক্ষার্থী
ইম্পেরিয়াল কলেজ লন্ডনের প্রাক্তন শিক্ষার্থী
লুকানো বিষয়শ্রেণী:
এইচকার্ডের সাথে নিবন্ধসমূহ
এ পৃষ্ঠায় শেষ পরিবর্তন হয়েছিল ১৫:০৮টার সময়, ২৩ আগস্ট ২০২৫ তারিখে।
ক্ষিতীশরঞ্জন চক্রবর্তী
৪টি ভাষা
আলোচনা যোগ করুন
অমলক রতন কোহলি
অমলক রতন কোহলি - উইকিপিডিয়া
সূচনা
১
তথ্যসূত্র
২
বহিঃসংযোগ
সূচিপত্র টগল করুন
অমলক রতন কোহলি
৩টি ভাষা
Eng

CONFIDENCE 1:
0.40822336077690125

CONTEXT 2:
১৯১৬-এ জন্ম
১৯৯৪-এ মৃত্যু
পশ্চিমবঙ্গের প্রকৌশলী
২০শ শতাব্দীর ভারতীয় উদ্ভাবক
২০শ শতাব্দীর ভারতীয় প্রকৌশলী
ভারতীয় পেটেন্ট ধারক
কলকাতা বিশ্ববিদ্যালয়ের প্রাক্তন শিক্ষার্থী
ইম্পেরিয়াল কলেজ লন্ডনের প্রাক্তন শিক্ষার্থী
লুকানো বিষয়শ্রেণী:
এইচকার্ডের সাথে নিবন্ধসমূহ
এ পৃষ্ঠায় শেষ পরিবর্তন হয়েছিল ১৫:০৮টার সময়, ২৩ আগস্ট ২০২৫ তারিখে।
ক্ষিতীশরঞ্জন চক্রবর্তী
৪টি ভাষা

In [6]:
# ==========================================================
# GEMMA GGUF QA HELPER
# RETURN FINAL ANSWER ONLY
# ==========================================================

def ask_gemma(prompt,
              max_tokens=40):

    if not globals().get('llm', None):
        raise Exception("Gemma GGUF not loaded")

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a ultra precise question answering assistant.\n"
                    "Think carefully before answering.\n"
                    "But output ONLY the final answer.\n"
                    "Do not show reasoning."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        top_p=1,
        max_tokens=max_tokens
    )

    raw_response = response[
        "choices"
    ][0]["message"]["content"]

    # --------------------------------
    # CLEAN OUTPUT
    # --------------------------------
    bad_strings = [
        "<think>",
        "</think>",
        "Thinking:",
        "Reasoning:",
        "Final Answer:",
        "Answer:",
        "উত্তর:"
    ]

    response = raw_response

    for s in bad_strings:
        response = response.replace(s, "")

    response = (
        response.strip()
        .split("\n")[0]
        .strip()
    )

    return response


print("✅ Gemma GGUF reasoning + clean final answer ready")

✅ Gemma GGUF reasoning + clean final answer ready


In [13]:
# MULTI-QUESTION SMOKE TEST (GEMMA GGUF)
# USE SAVED TEST RAG CSV
# RETRY ONLY IF MODEL SAYS CONTEXT MISSING
# ==========================================================

import torch
import gc
import pandas as pd

# ----------------------------------------------------------
# LOAD TEST WITH SAVED RAG
# ----------------------------------------------------------

TEST_PATH = "/kaggle/working/test_with_rag.csv"

test_df = pd.read_csv(TEST_PATH)

print("Loaded:", test_df.shape)
print(test_df.columns.tolist())

# ----------------------------------------------------------
# SELECT QUESTION IDS
# ----------------------------------------------------------

# qids = [210]

qids =[41, 99, 199, 210, 336, 349, 360, 434, 462, 480, 496, 502, 519, 636, 642, 691, 696, 822, 832, 893, 956, 1103, 1160, 1413]

# ==========================================================
# PROMPT TEMPLATE
# ==========================================================

def build_prompt(question, context):

    prompt = f"""


প্রশ্নের semantic meaning বুঝে Context থেকে সবচেয়ে উপযুক্ত ও নির্ভুল উত্তর বের করা। প্রশ্নের শব্দ বা বাক্যাংশ repeat করবে না — শুধুমাত্র answer part লিখবে।


গুরুত্বপূর্ণ:

• যদি প্রশ্ন specific নাম, ব্যক্তি, স্থান, তারিখ, সাল, সংখ্যা, পদবি, পুরস্কার, প্রতিষ্ঠান, ঘটনা বা নির্দিষ্ট তথ্য সম্পর্কে হয়, তাহলে শুধুমাত্র exact answer part লিখবে — অতিরিক্ত কোনো শব্দ, ব্যাখ্যা বা পূর্ণ বাক্য লিখবে না।

উদাহরণ:

Question:
বাংলাদেশের রাজধানী কী?

Bad Answer:
বাংলাদেশের রাজধানী ঢাকা।

Correct Answer:
ঢাকা

• যদি প্রশ্ন "কেন", "কারণ", "কীভাবে", "কিসের জন্য" ধরনের হয়, তাহলে পুরো semantic context বুঝে সবচেয়ে উপযুক্ত কারণ বা short reason phrase লিখবে — পূর্ণ বাক্য নয়, অতিরিক্ত ব্যাখ্যাও নয়।

Question:
রাহুলকে কেন স্কুল থেকে বের করে দেওয়া হয়েছিল?

Wrong Answer:
রাহুলকে কেন স্কুল থেকে বের করে দেওয়া হয়েছিল কারণ সে নকল করতে গিয়ে ধরা পড়ে।

Correct Answer:
নকল করতে গিয়ে ধরা পড়ায়


• যদি প্রশ্ন বৈজ্ঞানিক, চিকিৎসাবিজ্ঞান, অ্যানাটমি বা প্রযুক্তিগত বিষয়ের হয়, তাহলে Context-কে প্রধান উৎস হিসেবে ব্যবহার করবে এবং প্রয়োজন হলে সাধারণ বৈজ্ঞানিক জ্ঞান ব্যবহার করে সংক্ষিপ্ত ও function-based উত্তর দিবে — সংজ্ঞা বা অপ্রয়োজনীয় ব্যাখ্যা নয়।

Question:
ফুসফুসের প্রধান কাজ কী?

Wrong Answer:
ফুসফুস মানুষের শ্বাসযন্ত্রের একটি গুরুত্বপূর্ণ অঙ্গ।

Correct Answer:
অক্সিজেন গ্রহণ ও কার্বন ডাই-অক্সাইড ত্যাগ করা


• যদি প্রশ্ন কোনো ব্যক্তির স্বভাব, আচরণ বা বৈশিষ্ট্য সম্পর্কে হয় এবং Context-এ তা সাধারণীকরণ করা যায়, তাহলে সবচেয়ে natural short trait/behavior phrase লিখবে।
• উত্তর লেখার সময় সাধারণ বানান, তারিখ, মাস, সংখ্যা, সাল, ব্যক্তি/স্থানের নাম ও প্রচলিত বাংলা রূপ ঠিক রাখবে (যেমন: ফেব্রুয়ারি, জানুয়ারি, ডিসেম্বর, বঙ্গবন্ধু, চট্টগ্রাম)।তবে meaning পরিবর্তন করবে না।

example: অক্টোর → অক্টোবর


• প্রশ্নে যতটুকু চাওয়া হয়েছে ততটুকুই লিখবে — bracket এ অতিরিক্ত নাম, উদাহরণ, দেশ, details বা list expand করবে না, যদি প্রশ্নে explicitly তা না চাওয়া হয়।

Question:
কোন মহাদেশে মরুভূমি আছে?

Wrong Answer:
এশিয়া (সাহারা, গোবি, আরব মরুভূমি)

Wrong Answer:
এশিয়া, আফ্রিকা, অস্ট্রেলিয়া — বিভিন্ন দেশে মরুভূমি রয়েছে

Correct Answer:
এশিয়া, আফ্রিকা, অস্ট্রেলিয়া



• “কোথায়”, “কোন স্থানে”, “কোন অংশে”, “কোথায় পাওয়া যায়”, “কোথায় অবস্থিত” ধরনের প্রশ্নে যদি Context-এ বাস্তব/নির্দিষ্ট স্থান, অংশ, অধ্যায়, পরিশিষ্ট, প্রতিষ্ঠান, শহর, দেশ, ওয়েবসাইট বা location/entity স্পষ্টভাবে উল্লেখ থাকে, তাহলে সেটিই answer হিসেবে দিবে; “Context-এর অমুক লাইনে/অংশে আছে”, “উল্লেখ করা হয়েছে”, “দ্বিতীয় লাইনে আছে” ধরনের meta answer দিবে না।


Question:
জাতিসংঘের পরিবেশ সংক্রান্ত চুক্তিটি কোথায় উল্লেখ আছে?

Context:
জাতিসংঘের পরিবেশ সংক্রান্ত একটি বৈশ্বিক চুক্তি, যা ১৯৯২ সালে সাক্ষরিত হয়।

Wrong Answer:
Context-এ উল্লেখ আছে

Wrong Answer:
উপরের অংশে আছে

Correct Answer:
জাতিসংঘের পরিবেশ সংক্রান্ত একটি বৈশ্বিক চুক্তিতে





কঠোর নিয়ম:

1. শুধুমাত্র Context ব্যবহার করবে।
2. প্রশ্নের অর্থ গভীরভাবে বুঝবে।
3. পুরো Context বুঝে উত্তর দেবে।
4. উত্তর sentence আকারে লিখবে না।
5. শুধুমাত্র answer part / answer span লিখবে।
6. "ব্রিটিশ আমলে...", "প্রাপ্ত তথ্য অনুযায়ী...", "Context অনুযায়ী..." এভাবে শুরু করবে না।
7. অপ্রয়োজনীয় শব্দ যোগ করবে না।
8. প্রশ্ন repeat করবে না।
9. প্রশ্ন যদি "কেন" হয়, semantic context বুঝে সবচেয়ে উপযুক্ত কারণ বের করবে।
10. উত্তর সংক্ষিপ্ত, precise এবং natural হবে।
11. এক লাইনে শুধু final answer দিবে।
12. Context-এর বাইরে কিছু বানাবে না


খুব গুরুত্বপূর্ণ:
• যদি Context-এ উত্তর স্পষ্টভাবে না থাকে, তাহলে নিজের pretrained/general knowledge ব্যবহার করে সবচেয়ে সম্ভাব্য সংক্ষিপ্ত answer দিবে; “Context-এ উল্লেখ নেই”, “তথ্য নেই”, “উত্তর পাওয়া যায়নি”, “প্রদত্ত তথ্য অনুযায়ী” — এ ধরনের wording final answer-এ কখনো ব্যবহার করবে না।



Note: Context-এ "৪ টি ভাষা", "n টি ভাষা" এরকম Wikipedia থেকে direct copy করা translation option গুলো রয়ে যেতে পারে — এগুলো ignore করিও, এগুলো মূল তথ্য বা answer-এর অংশ না।


• Final answer এমনভাবে লিখবে যেন expected ground-truth answer-এর সাথে token overlap সর্বোচ্চ হয় — অতিরিক্ত token precision কমায়, missing token recall কমায়।
• সবচেয়ে ছোট কিন্তু সম্পূর্ণ semantic answer দিবে — extra explanation, prefix, suffix, bracket, উদাহরণ বা expansion যোগ করবে না।
• যদি answer এক বা কয়েকটি শব্দে দেওয়া সম্ভব হয়, তাহলে শুধু সেই minimal answer span লিখবে।
• multiple possible wording থাকলে সবচেয়ে standard, common এবং short wording ব্যবহার করবে



Context:
{context}

Question:
{question}

Answer:
"""
    return prompt


# ==========================================================
# CLEAN ANSWER
# ==========================================================

def clean_answer(output):

    return (
        str(output)
        .strip()
        .split("\n")[0]
        .replace("Final Answer:", "")
        .replace("উত্তর:", "")
        .strip()
    )


# ==========================================================
# RETRY LOGIC
# ==========================================================

def needs_retry(answer):

    answer = answer.lower().strip()

    retry_patterns = [
        "context",
        "প্রদত্ত context",
        "context-এ",
        "উল্লেখ নেই",
        "পাওয়া যায়নি",
        "তথ্য নেই",
        "তথ্য উল্লেখ নেই",
        "প্রসঙ্গে নেই"
    ]

    return any(
        p in answer
        for p in retry_patterns
    )


# ==========================================================
# LOOP THROUGH QUESTIONS
# ==========================================================

for qid in qids:

    row = test_df.iloc[qid]

    question = str(row["question"])

    print("\n" + "=" * 100)
    print(f"QUESTION ID: {qid}")
    print("=" * 100)

    print("\nQUESTION:")
    print(question)

    # ------------------------------------------------------
    # FIRST RETRIEVAL (SAVED CONTEXT1)
    # ------------------------------------------------------

    context = str(
        row["context1"]
    )

    confidence = float(
        row["confidence1"]
    )

    print("\nUsing saved context1...\n")

    print("=" * 80)
    print("CONFIDENCE:")
    print(f"{confidence:.4f}")

    print("\n" + "=" * 80)
    print("CONTEXT:")
    # print(context)
    print("=" * 80)

    # ------------------------------------------------------
    # BUILD PROMPT
    # ------------------------------------------------------

    prompt = build_prompt(
        question,
        context
    )

    max_tokens = 150

    # ------------------------------------------------------
    # GENERATE
    # ------------------------------------------------------

    print("\nGenerating with Gemma...\n")

    output = ask_gemma(
        prompt=prompt,
        max_tokens=max_tokens
    )

    answer = clean_answer(output)

    retried = False

    # ------------------------------------------------------
    # CONDITIONAL RETRY
    # USE SAVED CONTEXT2
    # ------------------------------------------------------

    if needs_retry(answer):

        retried = True

        print("\nRetry triggered...")
        print("Using saved context2...\n")

        context2 = str(
            row["context2"]
        )

        confidence2 = float(
            row["confidence2"]
        )

        print("\n" + "=" * 80)
        print("RETRY CONTEXT:")
        # print(context2)
        print("=" * 80)

        prompt2 = build_prompt(
            question,
            context2
        )

        output2 = ask_gemma(
            prompt=prompt2,
            max_tokens=max_tokens
        )

        answer2 = clean_answer(output2)

        # accept retry only if better
        if (
            len(answer2.strip()) > 0
            and not needs_retry(answer2)
        ):
            answer = answer2
            confidence = confidence2

    # ------------------------------------------------------
    # FINAL OUTPUT
    # ------------------------------------------------------

    print("=" * 80)
    print("RETRIED:")
    print(retried)

    print("\nFINAL ANSWER:")
    print(answer)

    print("=" * 80)

    gc.collect()
    torch.cuda.empty_cache()

Loaded: (1500, 6)
['index', 'question', 'context1', 'context2', 'confidence1', 'confidence2']

QUESTION ID: 41

QUESTION:
কেন উইলিয়ামসন টি২০আই ফর্ম্যাটে এখনো সেঞ্চুরি করেননি?

Using saved context1...

CONFIDENCE:
0.5303

CONTEXT:

Generating with Gemma...

RETRIED:
False

FINAL ANSWER:
তার সর্বোচ্চ স্কোর অপরাজিত ৭৩ রান হওয়ায়

QUESTION ID: 99

QUESTION:
বেলগাছিয়া পূর্ব বিধানসভা কেন্দ্র থেকে কতবার সিপিআই(এম)-এর লক্ষ্মীচরণ সেন নির্বাচিত হন?

Using saved context1...

CONFIDENCE:
0.8318

CONTEXT:

Generating with Gemma...

RETRIED:
False

FINAL ANSWER:
৩ বার

QUESTION ID: 199

QUESTION:
রমনীলাল কিরচাঁদ গান্ধীর পিতা কে ছিলেন?

Using saved context1...

CONFIDENCE:
0.7080

CONTEXT:

Generating with Gemma...


Retry triggered...
Using saved context2...


RETRY CONTEXT:
RETRIED:
True

FINAL ANSWER:
কিরচাঁদ গান্ধী

QUESTION ID: 210

QUESTION:
কিশোরগঞ্জ শব্দটির অর্থ কী?

Using saved context1...

CONFIDENCE:
0.3767

CONTEXT:

Generating with Gemma...

RETRIED:
False

FINAL ANSWER:
কিশোর মোহন বসাক

In [15]:
# ==========================================================
# FULL TEST INFERENCE (GEMMA GGUF)
# USE SAVED TEST RAG CSV
# context1 -> retry with context2
# SAVE AS submission10.csv
# ==========================================================

import torch
import gc
import pandas as pd
from tqdm.auto import tqdm

# ----------------------------------------------------------
# LOAD TEST WITH SAVED RAG
# ----------------------------------------------------------

TEST_PATH = "/kaggle/working/test_with_rag.csv"

test_df = pd.read_csv(TEST_PATH)

print("Loaded:", test_df.shape)
print(test_df.columns.tolist())

print(
    "Total test questions:",
    len(test_df)
)

# ==========================================================
# PROMPT TEMPLATE
# ==========================================================

def build_prompt(question, context):

    prompt = f"""

প্রশ্নের semantic meaning বুঝে Context থেকে সবচেয়ে উপযুক্ত ও নির্ভুল উত্তর বের করা। প্রশ্নের শব্দ বা বাক্যাংশ repeat করবে না — শুধুমাত্র answer part লিখবে।

গুরুত্বপূর্ণ:

• যদি প্রশ্ন specific নাম, ব্যক্তি, স্থান, তারিখ, সাল, সংখ্যা, পদবি, পুরস্কার, প্রতিষ্ঠান, ঘটনা বা নির্দিষ্ট তথ্য সম্পর্কে হয়, তাহলে শুধুমাত্র exact answer part লিখবে — অতিরিক্ত কোনো শব্দ, ব্যাখ্যা বা পূর্ণ বাক্য লিখবে না।

উদাহরণ:

Question:
বাংলাদেশের রাজধানী কী?

Bad Answer:
বাংলাদেশের রাজধানী ঢাকা।

Correct Answer:
ঢাকা

• যদি প্রশ্ন "কেন", "কারণ", "কীভাবে", "কিসের জন্য" ধরনের হয়, তাহলে পুরো semantic context বুঝে সবচেয়ে উপযুক্ত কারণ বা short reason phrase লিখবে — পূর্ণ বাক্য নয়, অতিরিক্ত ব্যাখ্যাও নয়।

Question:
রাহুলকে কেন স্কুল থেকে বের করে দেওয়া হয়েছিল?

Wrong Answer:
রাহুলকে কেন স্কুল থেকে বের করে দেওয়া হয়েছিল কারণ সে নকল করতে গিয়ে ধরা পড়ে।

Correct Answer:
নকল করতে গিয়ে ধরা পড়ায়

• যদি প্রশ্ন বৈজ্ঞানিক, চিকিৎসাবিজ্ঞান, অ্যানাটমি বা প্রযুক্তিগত বিষয়ের হয়, তাহলে Context-কে প্রধান উৎস হিসেবে ব্যবহার করবে এবং প্রয়োজন হলে সাধারণ বৈজ্ঞানিক জ্ঞান ব্যবহার করে সংক্ষিপ্ত ও function-based উত্তর দিবে — সংজ্ঞা বা অপ্রয়োজনীয় ব্যাখ্যা নয়।

Question:
ফুসফুসের প্রধান কাজ কী?

Wrong Answer:
ফুসফুস মানুষের শ্বাসযন্ত্রের একটি গুরুত্বপূর্ণ অঙ্গ।

Correct Answer:
অক্সিজেন গ্রহণ ও কার্বন ডাই-অক্সাইড ত্যাগ করা

• যদি প্রশ্ন কোনো ব্যক্তির স্বভাব, আচরণ বা বৈশিষ্ট্য সম্পর্কে হয় এবং Context-এ তা সাধারণীকরণ করা যায়, তাহলে সবচেয়ে natural short trait/behavior phrase লিখবে।

• উত্তর লেখার সময় সাধারণ বানান, তারিখ, মাস, সংখ্যা, সাল, ব্যক্তি/স্থানের নাম ও প্রচলিত বাংলা রূপ ঠিক রাখবে (যেমন: ফেব্রুয়ারি, জানুয়ারি, ডিসেম্বর, বঙ্গবন্ধু, চট্টগ্রাম)। তবে meaning পরিবর্তন করবে না।

example:
অক্টোর → অক্টোবর

• প্রশ্নে যতটুকু চাওয়া হয়েছে ততটুকুই লিখবে — bracket এ অতিরিক্ত নাম, উদাহরণ, দেশ, details বা list expand করবে না, যদি প্রশ্নে explicitly তা না চাওয়া হয়।

Question:
কোন মহাদেশে মরুভূমি আছে?

Wrong Answer:
এশিয়া (সাহারা, গোবি, আরব মরুভূমি)

Wrong Answer:
এশিয়া, আফ্রিকা, অস্ট্রেলিয়া — বিভিন্ন দেশে মরুভূমি রয়েছে

Correct Answer:
এশিয়া, আফ্রিকা, অস্ট্রেলিয়া

• “কোথায়”, “কোন স্থানে”, “কোন অংশে”, “কোথায় পাওয়া যায়”, “কোথায় অবস্থিত” ধরনের প্রশ্নে যদি Context-এ বাস্তব/নির্দিষ্ট স্থান, অংশ, অধ্যায়, পরিশিষ্ট, প্রতিষ্ঠান, শহর, দেশ, ওয়েবসাইট বা location/entity স্পষ্টভাবে উল্লেখ থাকে, তাহলে সেটিই answer হিসেবে দিবে; “Context-এর অমুক লাইনে/অংশে আছে”, “উল্লেখ করা হয়েছে”, “দ্বিতীয় লাইনে আছে” ধরনের meta answer দিবে না।

কঠোর নিয়ম:

1. শুধুমাত্র Context ব্যবহার করবে।
2. প্রশ্নের অর্থ গভীরভাবে বুঝবে।
3. পুরো Context বুঝে উত্তর দেবে।
4. উত্তর sentence আকারে লিখবে না।
5. শুধুমাত্র answer part / answer span লিখবে।
6. "ব্রিটিশ আমলে...", "প্রাপ্ত তথ্য অনুযায়ী...", "Context অনুযায়ী..." এভাবে শুরু করবে না।
7. অপ্রয়োজনীয় শব্দ যোগ করবে না।
8. প্রশ্ন repeat করবে না।
9. প্রশ্ন যদি "কেন" হয়, semantic context বুঝে সবচেয়ে উপযুক্ত কারণ বের করবে।
10. উত্তর সংক্ষিপ্ত, precise এবং natural হবে।
11. এক লাইনে শুধু final answer দিবে।
12. Context-এর বাইরে কিছু বানাবে না।

খুব গুরুত্বপূর্ণ:

• যদি Context-এ উত্তর স্পষ্টভাবে না থাকে, তাহলে নিজের pretrained/general knowledge ব্যবহার করে সবচেয়ে সম্ভাব্য সংক্ষিপ্ত answer দিবে; “Context-এ উল্লেখ নেই”, “তথ্য নেই”, “উত্তর পাওয়া যায়নি”, “প্রদত্ত তথ্য অনুযায়ী” — এ ধরনের wording final answer-এ কখনো ব্যবহার করবে না।

• Final answer এমনভাবে লিখবে যেন expected ground-truth answer-এর সাথে token overlap সর্বোচ্চ হয় — অতিরিক্ত token precision কমায়, missing token recall কমায়।

• সবচেয়ে ছোট কিন্তু সম্পূর্ণ semantic answer দিবে — extra explanation, prefix, suffix, bracket, উদাহরণ বা expansion যোগ করবে না।

• যদি answer এক বা কয়েকটি শব্দে দেওয়া সম্ভব হয়, তাহলে শুধু সেই minimal answer span লিখবে।

• multiple possible wording থাকলে সবচেয়ে standard, common এবং short wording ব্যবহার করবে।

Note:
Context-এ "৪ টি ভাষা", "n টি ভাষা" এরকম Wikipedia translation noise থাকতে পারে — এগুলো ignore করিও।

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt


# ==========================================================
# CLEAN ANSWER
# ==========================================================

def clean_answer(output):

    answer = (
        str(output)
        .strip()
        .split("\n")[0]
        .replace("Final Answer:", "")
        .replace("উত্তর:", "")
        .strip()
    )

    return answer


# ==========================================================
# RETRY LOGIC
# ==========================================================

def needs_retry(answer):

    answer = answer.lower().strip()

    retry_patterns = [
        "context",
        "context-এ",
        "উল্লেখ নেই",
        "তথ্য নেই",
        "পাওয়া যায়নি",
        "প্রদত্ত তথ্য",
        "উত্তর পাওয়া যায়নি",
        "জানা যায়নি",
    ]

    return any(
        p in answer
        for p in retry_patterns
    )


# ==========================================================
# STORE ANSWERS
# ==========================================================

predictions = []

# ==========================================================
# LOOP THROUGH TEST
# ==========================================================

for idx in tqdm(
    range(len(test_df)),
    desc="Inference"
):

    try:

        row = test_df.iloc[idx]

        question = str(
            row["question"]
        ).strip()

        # --------------------------------------------------
        # CONTEXT1 (SAVED)
        # --------------------------------------------------

        context = str(
            row["context1"]
        )

        confidence = float(
            row["confidence1"]
        )

        prompt = build_prompt(
            question,
            context
        )

        max_tokens = 150

        # --------------------------------------------------
        # GENERATE
        # --------------------------------------------------

        output = ask_gemma(
            prompt=prompt,
            max_tokens=max_tokens
        )

        answer = clean_answer(output)

        retried = False

        # --------------------------------------------------
        # RETRY WITH CONTEXT2
        # --------------------------------------------------

        if needs_retry(answer):

            retried = True

            context2 = str(
                row["context2"]
            )

            confidence2 = float(
                row["confidence2"]
            )

            prompt2 = build_prompt(
                question,
                context2
            )

            output2 = ask_gemma(
                prompt=prompt2,
                max_tokens=max_tokens
            )

            answer2 = clean_answer(
                output2
            )

            if (
                len(answer2.strip()) > 0
                and not needs_retry(answer2)
            ):
                answer = answer2
                confidence = confidence2

        predictions.append(answer)

        # --------------------------------------------------
        # SHOW FIRST 10
        # --------------------------------------------------

        if idx < 10:

            print("\n" + "=" * 100)
            print(f"QUESTION {idx+1}")
            print("=" * 100)

            print("\nQUESTION:")
            print(question)

            print("\nCONFIDENCE:")
            print(f"{confidence:.4f}")

            print("\nRETRIED:")
            print(retried)

            print("\nFINAL ANSWER:")
            print(answer)

        # --------------------------------------------------
        # TEMP SAVE
        # --------------------------------------------------

        if (idx + 1) % 50 == 0:

            temp_submission = pd.DataFrame({
                "index":
                    test_df["index"][:len(predictions)],
                "answer":
                    predictions
            })

            temp_submission.to_csv(
                "submission_temp10.csv",
                index=False
            )

            print(
                f"\n✅ Saved progress: "
                f"{idx+1}/{len(test_df)}"
            )

        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:

        print(
            f"\n❌ Error at index {idx}"
        )
        print(e)

        predictions.append("")

# ==========================================================
# FINAL SUBMISSION
# ==========================================================

submission = pd.DataFrame({
    "index": test_df["index"],
    "answer": predictions
})

submission.to_csv(
    "submission11.csv",
    index=False
)

print("\n✅ Saved: submission11.csv")
print(submission.head(10))

Loaded: (1500, 6)
['index', 'question', 'context1', 'context2', 'confidence1', 'confidence2']
Total test questions: 1500


Inference:   0%|          | 0/1500 [00:00<?, ?it/s]


QUESTION 1

QUESTION:
অমলক রতন কোহলি কোন ক্ষেত্রে বিশিষ্ট হিসেবে সম্মানিত?

CONFIDENCE:
0.4082

RETRIED:
False

FINAL ANSWER:
শিক্ষাবিদ এবং মানব সম্পদ প্রশিক্ষক

QUESTION 2

QUESTION:
ম্যাক্স ব্যারনের বন্ধুরা তাকে কেন প্রায়ই ঠাট্টা-বিদ্রুপ করে?

CONFIDENCE:
0.2118

RETRIED:
False

FINAL ANSWER:
তার প্রাথমিক চরিত্রের কারণে (নীতিগতভাবে খুঁতখুঁতে হওয়া)

QUESTION 3

QUESTION:
ব্রিটিশ আমলে ব্যাঙ্কশাল কোর্ট কী নামে পরিচিত ছিল?

CONFIDENCE:
0.3803

RETRIED:
False

FINAL ANSWER:
স্মল মোজেস কোর্ট (Small Causes Court)

QUESTION 4

QUESTION:
কে জুতা বানানোর আধুনিক যন্ত্র তৈরি করেন?

CONFIDENCE:
0.5511

RETRIED:
False

FINAL ANSWER:
লেম্যান আর. ব্লেক

QUESTION 5

QUESTION:
সার্ভাইভার সিরিজের ম্যাচে কখনও কখনও কী ধরনের অতিরিক্ত শর্ত আরোপ করা হয়?

CONFIDENCE:
0.9998

RETRIED:
False

FINAL ANSWER:
পরাজিত দলের সদস্যদের ডাব্লিউডাব্লিউই হতে বরখাস্ত করা হবে

QUESTION 6

QUESTION:
আফরোজা পারভীন কবে বাংলা একাডেমি পুরস্কার অর্জন করেন?

CONFIDENCE:
0.5107

RETRIED:
False

FINAL ANSWER:
২০২৩

QUESTION 7

QU